#### Updated Vibrometer Experiment
- Goal: Estimate transfer function for system

In [ ]:
import librosa
import numpy as np
import scipy.signal as signal
import sounddevice as sd
import matplotlib.pyplot as plt
import pandas as pd
from math import gcd
import scipy.fft
from scipy.signal import welch, csd, coherence
from scipy.signal import butter, sosfilt

##### Input Data:
- First: try with square wave
- Second (if time): try with music

In [ ]:
# First - Try with square wave
# Generate square wave:
fs = 44100  # Sample rate. Note: rode wireless pro can export recordings with either fs= 48 kHz or 44.1 kHz
duration = 10  # seconds
t = np.linspace(0, duration, int(fs * duration))
freq = 100  # Hz
square_wave = signal.square(2 * np.pi * freq * t)

# Play and record
#recording = sd.playrec(square_wave, fs, channels=1)
#sd.wait()

# Plot input signal:
plt.figure(figsize=(12,2))
plt.plot(t[:1000], square_wave[:1000])
#plt.xlim(right=1)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude') # units?
plt.title('Input Signal (Square Wave)')
plt.grid(True, alpha=0.3)
plt.show()

***
##### Load Measurement Data:

In [ ]:
# Vibrometer Measurements:
column_names1= ['Time1 (s)', 'Signal1 (m)']
background = pd.read_table("background.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names1)
#background.head()

column_names2= ['Time2 (s)', 'Signal2 (m)']
no_glass = pd.read_table("test no glass.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names2)
#no_glass.head()

# Run A measurements:~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
column_names1a= ['Time1a (s)', 'Signal1a (m)']
run_1a = pd.read_table("run 1a (ver.3).txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names1a) # re-recorded due to signal jump
#run_1a.head()

column_names2a = ['Time2a (s)', 'Signal2a (m)']
run_2a = pd.read_table("run 2a (20cm from speaker).txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names2a) 
#print(run_2a.head())

column_names3a = ['Time3a (s)', 'Signal3a (m)']
run_3a = pd.read_table("run 3a (ver3) use this one.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names3a) 
#print(run_3a.head())

column_names4a = ['Time4a (s)', 'Signal4a (m)']
run_4a = pd.read_table("run 4a (glass 40cm from speaker).txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names4a) 
#print(run_4a.head())

column_names5a = ['Time5a (s)', 'Signal5a (m)']
run_5a = pd.read_table("run 5a (ver2).txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names5a) 
#print(run_5a.head())

# Run B measurements:~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
column_names2b = ['Time2b (s)', 'Signal2b (m)']
run_2b = pd.read_table("run 2b (speaker side).txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names2b)
#print(run_2b.head())

column_names3b = ['Time3b (s)', 'Signal3b (m)']
run_3b = pd.read_table("run 3b (ver2).txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names3b)
#print(run_3b.head())

column_names4b = ['Time4b (s)', 'Signal4b (m)']
run_4b = pd.read_table("run 4b.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names4b)
#print(run_4b.head())

column_names5b = ['Time5b (s)', 'Signal5b (m)']
run_5b = pd.read_table("run 5b.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names5b)
#print(run_5b.head())

In [ ]:
# Sample rate test (to check vibrometer settings):
print("Measurement sample rate in Hz:", int(len(background['Time1 (s)'])/background['Time1 (s)'].iloc[-1]) )# samples/second

In [ ]:
# Microphone Recordings:
# WirelessPRO-ref folder:~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("WirelessPRO-ref folder:")
# 00007_WirelessPRO-ref.wav
# specify file path if recordings are in a different folder from this notebook:
file_path_07 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00007_WirelessPRO-ref.wav"
y_07, sr_07 = librosa.load(file_path_07, sr=None) # sr=None keeps original sample rate (44100 Hz)
print(f"07 Sample rate: {sr_07}")
print(f"07 Duration: {len(y_07)/sr_07:.2f} seconds")

# 00008_WirelessPRO-ref.wav
file_path_08 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00008_WirelessPRO-ref.wav"
y_08, sr_08 = librosa.load(file_path_08, sr=None)
print(f"08 Sample rate: {sr_08}")
print(f"08 Duration: {len(y_08)/sr_08:.2f} seconds")

# 00009_WirelessPRO-ref.wav
file_path_09 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00009_WirelessPRO-ref.wav"
y_09, sr_09 = librosa.load(file_path_09, sr=None)
print(f"09 Sample rate: {sr_09}")
print(f"09 Duration: {len(y_09)/sr_09:.2f} seconds")

# 00010_WirelessPRO-ref.wav
file_path_10 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00010_WirelessPRO-ref.wav"
y_10, sr_10 = librosa.load(file_path_10, sr=None)
print(f"10 Sample rate: {sr_10}")
print(f"10 Duration: {len(y_10)/sr_10:.2f} seconds")

# 00011_WirelessPRO-ref.wav
file_path_11 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00011_WirelessPRO-ref.wav"
y_11, sr_11 = librosa.load(file_path_11, sr=None)
print(f"11 Sample rate: {sr_11}")
print(f"11 Duration: {len(y_11)/sr_11:.2f} seconds")

# 00012_WirelessPRO-ref.wav
file_path_12 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00012_WirelessPRO-ref.wav"
y_12, sr_12 = librosa.load(file_path_12, sr=None)
print(f"12 Sample rate: {sr_12}")
print(f"12 Duration: {len(y_12)/sr_12:.2f} seconds")

# 00013_WirelessPRO-ref.wav
file_path_13 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00013_WirelessPRO-ref.wav"
y_13, sr_13 = librosa.load(file_path_13, sr=None)
print(f"13 Sample rate: {sr_13}")
print(f"13 Duration: {len(y_13)/sr_13:.2f} seconds")

# 00014_WirelessPRO-ref.wav
file_path_14 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00014_WirelessPRO-ref.wav"
y_14, sr_14 = librosa.load(file_path_14, sr=None)
print(f"14 Sample rate: {sr_14}")
print(f"14 Duration: {len(y_14)/sr_14:.2f} seconds")

# 00015_WirelessPRO-ref.wav
file_path_15 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00015_WirelessPRO-ref.wav"
y_15, sr_15 = librosa.load(file_path_15, sr=None)
print(f"15 Sample rate: {sr_15}")
print(f"15 Duration: {len(y_15)/sr_15:.2f} seconds")

# 00016_WirelessPRO-ref.wav
file_path_16 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00016_WirelessPRO-ref.wav"
y_16, sr_16 = librosa.load(file_path_16, sr=None)
print(f"16 Sample rate: {sr_16}")
print(f"16 Duration: {len(y_16)/sr_16:.2f} seconds")

# 00017_WirelessPRO-ref.wav
file_path_17 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00017_WirelessPRO-ref.wav"
y_17, sr_17 = librosa.load(file_path_17, sr=None)
print(f"17 Sample rate: {sr_17}")
print(f"17 Duration: {len(y_17)/sr_17:.2f} seconds")

# 00018_WirelessPRO-ref.wav
file_path_18 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00018_WirelessPRO-ref.wav"
y_18, sr_18 = librosa.load(file_path_18, sr=None)
print(f"18 Sample rate: {sr_18}")
print(f"18 Duration: {len(y_18)/sr_18:.2f} seconds")

# 00019_WirelessPRO-ref.wav
file_path_19 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00019_WirelessPRO-ref.wav"
y_19, sr_19 = librosa.load(file_path_19, sr=None)
print(f"19 Sample rate: {sr_19}")
print(f"19 Duration: {len(y_19)/sr_19:.2f} seconds")

# 00020_WirelessPRO-ref.wav
file_path_20 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-ref\00020_WirelessPRO-ref.wav"
y_20, sr_20 = librosa.load(file_path_20, sr=None)
print(f"20 Sample rate: {sr_20}")
print(f"20 Duration: {len(y_20)/sr_20:.2f} seconds")

print("")
# WirelessPRO-roving folder:~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("WirelessPRO-roving folder:")
# 00008_Wireless PRO.wav
path_08 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00008_Wireless PRO.wav"
y08, sr08 = librosa.load(path_08, sr=None)
print(f"08 Sample rate: {sr08}")
print(f"08 Duration: {len(y08)/sr08:.2f} seconds")

# 00009_Wireless PRO.wav
path_09 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00009_Wireless PRO.wav"
y09, sr09 = librosa.load(path_09, sr=None)
print(f"09 Sample rate: {sr09}")
print(f"09 Duration: {len(y09)/sr09:.2f} seconds")

# 00010_Wireless PRO.wav
path_10 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00010_Wireless PRO.wav"
y10, sr10 = librosa.load(path_10, sr=None)
print(f"10 Sample rate: {sr10}")
print(f"10 Duration: {len(y10)/sr10:.2f} seconds")

# 00011_Wireless PRO.wav
path_11 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00011_Wireless PRO.wav"
y11, sr11 = librosa.load(path_11, sr=None)
print(f"11 Sample rate: {sr11}")
print(f"11 Duration: {len(y11)/sr11:.2f} seconds")

# 00012_Wireless PRO.wav
path_12 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00012_Wireless PRO.wav"
y12, sr12 = librosa.load(path_12, sr=None)
print(f"12 Sample rate: {sr12}")
print(f"12 Duration: {len(y12)/sr12:.2f} seconds")

# 00013_Wireless PRO.wav
path_13 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00013_Wireless PRO.wav"
y13, sr13 = librosa.load(path_13, sr=None)
print(f"13 Sample rate: {sr13}")
print(f"13 Duration: {len(y13)/sr13:.2f} seconds")

# 00014_Wireless PRO.wav
path_14 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00014_Wireless PRO.wav"
y14, sr14 = librosa.load(path_14, sr=None)
print(f"14 Sample rate: {sr14}")
print(f"14 Duration: {len(y14)/sr14:.2f} seconds")

# 00015_Wireless PRO.wav
path_15 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00015_Wireless PRO.wav"
y15, sr15 = librosa.load(path_15, sr=None)
print(f"15 Sample rate: {sr15}")
print(f"15 Duration: {len(y15)/sr15:.2f} seconds")

# 00016_Wireless PRO.wav
path_16 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00016_Wireless PRO.wav"
y16, sr16 = librosa.load(path_16, sr=None)
print(f"16 Sample rate: {sr16}")
print(f"16 Duration: {len(y16)/sr16:.2f} seconds")

# 00017_Wireless PRO.wav
path_17 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00017_Wireless PRO.wav"
y17, sr17 = librosa.load(path_17, sr=None)
print(f"17 Sample rate: {sr17}")
print(f"17 Duration: {len(y17)/sr17:.2f} seconds")

# 00018_Wireless PRO.wav
path_18 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00018_Wireless PRO.wav"
y18, sr18 = librosa.load(path_18, sr=None)
print(f"18 Sample rate: {sr18}")
print(f"18 Duration: {len(y18)/sr18:.2f} seconds")

# 00019_Wireless PRO.wav
path_19 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00019_Wireless PRO.wav"
y19, sr19 = librosa.load(path_19, sr=None)
print(f"19 Sample rate: {sr19}")
print(f"19 Duration: {len(y19)/sr19:.2f} seconds")

# 00020_Wireless PRO.wav
path_20 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00020_Wireless PRO.wav"
y20, sr20 = librosa.load(path_20, sr=None)
print(f"20 Sample rate: {sr20}")
print(f"20 Duration: {len(y20)/sr20:.2f} seconds")

# 00021_Wireless PRO.wav
path_21 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00021_Wireless PRO.wav"
y21, sr21 = librosa.load(path_21, sr=None)
print(f"21 Sample rate: {sr21}")
print(f"21 Duration: {len(y21)/sr21:.2f} seconds")

# 00022_Wireless PRO.wav
path_22 = r"C:\Users\Owner\OneDrive - University of Waterloo\Co-op\Research Assistant\Vibrometer Experiment\transfer-function-updates\WirelessPRO-roving\00022_Wireless PRO.wav"
y22, sr22 = librosa.load(path_22, sr=None)
print(f"22 Sample rate: {sr22}")
print(f"22 Duration: {len(y22)/sr22:.2f} seconds")


In [ ]:
# Matching recordings to vibrometer measurements...
#Run 1a (ver.3) --> ref 08, rov 10
#Run 2a --> ref 09, rov 11
#Run 2b --> ref 10, rov 12
#Run 3a --> ref 13, rov 15
#Run 3b --> ref 15, rov 17
#Run 4a --> ref 16, rov 18
#Run 4b --> ref 17, rov 19
#Run 5a --> ref 19, rov 21
#Run 5b --> ref 20, rov 22
# make dataframes for the recordings:
# Run 1a:
# ref 08
len_08= len(y_08)
time_08 = librosa.samples_to_time(np.arange(len_08), sr=44100)
ref_08 = pd.DataFrame({'Amplitude': y_08, 'Time (s)': time_08})
# rov 10
len10 = len(y10)
time10 = librosa.samples_to_time(np.arange(len10), sr=44100)
rov_10 = pd.DataFrame({'Amplitude': y10, 'Time (s)': time10})

# Run 2a:
# ref 09
len_09 = len(y_09)
time_09 = librosa.samples_to_time(np.arange(len_09), sr=44100)
ref_09 = pd.DataFrame({'Amplitude': y_09, 'Time (s)': time_09})
# rov 11
len11 = len(y11)
time11 = librosa.samples_to_time(np.arange(len11), sr=44100)
rov_11 = pd.DataFrame({'Amplitude': y11, 'Time (s)': time11})

# Run 2b:
# ref 10
len_10 = len(y_10)
time_10 = librosa.samples_to_time(np.arange(len_10), sr=44100)
ref_10 = pd.DataFrame({'Amplitude': y_10, 'Time (s)': time_10})
# rov 12
len12 = len(y12)
time12 = librosa.samples_to_time(np.arange(len12), sr=44100)
rov_12 = pd.DataFrame({'Amplitude': y12, 'Time (s)': time12})

# Run 3a:
# ref 13
len_13 = len(y_13)
time_13 = librosa.samples_to_time(np.arange(len_13), sr=44100)
ref_13 = pd.DataFrame({'Amplitude': y_13, 'Time (s)': time_13})
# rov 15
len15 = len(y15)
time15 = librosa.samples_to_time(np.arange(len15), sr=44100)
rov_15 = pd.DataFrame({'Amplitude': y15, 'Time (s)': time15})

# Run 3b:
# ref 15
len_15 = len(y_15)
time_15 = librosa.samples_to_time(np.arange(len_15), sr=44100)
ref_15 = pd.DataFrame({'Amplitude': y_15, 'Time (s)': time_15})
# rov 17
len17 = len(y17)
time17 = librosa.samples_to_time(np.arange(len17), sr=44100)
rov_17 = pd.DataFrame({'Amplitude': y17, 'Time (s)': time17})

# Run 4a:
# ref 16
len_16 = len(y_16)
time_16 = librosa.samples_to_time(np.arange(len_16), sr=44100)
ref_16 = pd.DataFrame({'Amplitude': y_16, 'Time (s)': time_16})
# rov 18
len18 = len(y18)
time18 = librosa.samples_to_time(np.arange(len18), sr=44100)
rov_18 = pd.DataFrame({'Amplitude': y18, 'Time (s)': time18})

# Run 4b:
#ref 17
len_17 = len(y_17)
time_17 = librosa.samples_to_time(np.arange(len_17), sr=44100)
ref_17 = pd.DataFrame({'Amplitude': y_17, 'Time (s)': time_17})
#rov 19
len19 = len(y19)
time19 = librosa.samples_to_time(np.arange(len19), sr=44100)
rov_19 = pd.DataFrame({'Amplitude': y19, 'Time (s)': time19})

# Run 5a:
# ref 19
len_19 = len(y_19)
time_19 = librosa.samples_to_time(np.arange(len_19), sr=44100)
ref_19 = pd.DataFrame({'Amplitude': y_19, 'Time (s)': time_19})
# rov 21
len21 = len(y21)
time21 = librosa.samples_to_time(np.arange(len21), sr=44100)
rov_21 = pd.DataFrame({'Amplitude': y21, 'Time (s)': time21})

# Run 5b:
# ref 20
len_20 = len(y_20)
time_20 = librosa.samples_to_time(np.arange(len_20), sr=44100)
ref_20 = pd.DataFrame({'Amplitude': y_20, 'Time (s)': time_20})
# rov 22
len22 = len(y22)
time22 = librosa.samples_to_time(np.arange(len22), sr=44100)
rov_22 = pd.DataFrame({'Amplitude': y22, 'Time (s)': time22})
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Put in lists for plotting:
run_A_mic_ref = [ref_08, ref_09, ref_13, ref_16, ref_19] # all run A reference measurements
run_A_mic_rov = [rov_10, rov_11, rov_15, rov_18, rov_21] # all run A roving measurements
run_A_all = [ref_08, rov_10, ref_09, rov_11, ref_13, rov_15, ref_16, rov_18, ref_19, rov_21] # ref + rov, ordered by run A number

run_B_mic_ref = [ref_10, ref_15, ref_17, ref_20] # all run B reference measurements
run_B_mic_rov = [rov_12, rov_17, rov_19, rov_22] # all run B roving measurements
run_B_all = [ref_10, rov_12, ref_15, rov_17, ref_17, rov_19, ref_20, rov_22] # ref + rov, ordered by run B number

***
##### Plot Raw Data:

In [ ]:
# VIBROMETER:
# initial measurements: "background" and "no_glass"
run_A_measurements = [run_1a, run_2a, run_3a, run_4a, run_5a]
run_B_measurements = [run_2b, run_3b, run_4b, run_5b]

# Plot initial measurements first:
plt.figure(figsize=(12,4))
plt.plot(background['Time1 (s)'], background['Signal1 (m)'])
plt.grid(True, alpha=0.5)
plt.title("Background Measurement")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (m)")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(no_glass['Time2 (s)'], no_glass['Signal2 (m)'])
plt.grid(True, alpha=0.5)
plt.title("Square Wave Measurement (No Glass)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (m)")
plt.show()

for i, dframe in enumerate(run_A_measurements):
    time_col_a = f"Time{i+1}a (s)"
    signal_col_a = f"Signal{i+1}a (m)"
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col_a], dframe[signal_col_a])
    plt.grid(True, alpha=0.5)
    plt.title(f"Measurement {i+1}a")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    plt.show()

for i, dframe in enumerate(run_B_measurements):
    time_col_b = f"Time{i+2}b (s)"
    signal_col_b = f"Signal{i+2}b (m)"
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col_b], dframe[signal_col_b])
    plt.grid(True, alpha=0.5)
    plt.title(f"Measurement {i+2}b")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    plt.show()
    

In [ ]:
# MICROPHONES:
#run_A_mic_ref = [ref_08, ref_09, ref_13, ref_16, ref_19] # all run A reference measurements
#run_A_mic_rov = [rov_10, rov_11, rov_15, rov_18, rov_21] # all run A roving measurements
#run_B_mic_ref = [ref_10, ref_15, ref_17, ref_20] # all run B reference measurements
#run_B_mic_rov = [rov_12, rov_17, rov_19, rov_22] # all run B roving measurements

print("Run A recordings:")
for i, (ref, rov) in enumerate(zip(run_A_mic_ref, run_A_mic_rov)):
    signal_col = 'Amplitude'
    time_col= 'Time (s)'
    
    # Plot reference:
    plt.figure(figsize=(12,4))
    plt.plot(ref[time_col], ref[signal_col])
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.ylim(-0.1, 0.1)
    plt.title(f"Reference Recording {i+1}a")
    plt.show()
    
    # Plot roving:
    plt.figure(figsize=(12,4))
    plt.plot(rov[time_col], rov[signal_col])
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.ylim(-0.1, 0.1)
    plt.title(f"Roving Recording {i+1}a")
    plt.show()

print("Run B recordings:")
for i, (ref, rov) in enumerate(zip(run_B_mic_ref, run_B_mic_rov)):
    signal_col = 'Amplitude'
    time_col = 'Time (s)'

    # Plot reference:
    plt.figure(figsize=(12,4))
    plt.plot(ref[time_col], ref[signal_col])
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.ylim(-0.1, 0.1)
    plt.title(f"Reference Recording {i+2}b")
    plt.show()

    # Plot roving:
    plt.figure(figsize=(12,4))
    plt.plot(rov[time_col], rov[signal_col])
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.ylim(-0.1, 0.1)
    plt.title(f"Roving Recording {i+2}b")
    plt.show()

# need to trim measurements to where square wave is played
# also trim spikes in amplitude from turning the mics on/off

In [ ]:
# need to trim measurements to where square wave is played
# also trim spikes in amplitude from turning the mics on/off

# Recording A's~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Ref recording 1a: ref_08
# drop beginning and ending values:
ref_08 = ref_08[(ref_08['Time (s)'] > 1.6) & (ref_08['Time (s)'] < 12.5)]
# check with plot:
plt.figure(figsize=(12,4))
plt.plot(ref_08['Time (s)'], ref_08['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Reference Recording 1a")
plt.ylim(-0.1, 0.1)
#plt.xlim(left=12)
plt.show()
# Rov recording 1a: rov_10
rov_10 = rov_10[(rov_10['Time (s)'] > 0.1)]
plt.figure(figsize=(12,4))
plt.plot(rov_10['Time (s)'], rov_10['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Roving Recording 1a")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=1)
plt.show()

# Ref recording 2a: ref 09
ref_09 = ref_09[(ref_09['Time (s)'] > 1.3)]
plt.figure(figsize=(12,4))
plt.plot(ref_09['Time (s)'], ref_09['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Reference Recording 2a")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=1.5)
plt.show()
# Rov recording 2a: rov 11
rov_11 = rov_11[(rov_11['Time (s)'] > 1) & (rov_11['Time (s)'] < 8.88)]
plt.figure(figsize=(12,4))
plt.plot(rov_11['Time (s)'], rov_11['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Roving Recording 2a")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=1)
plt.show()

# Ref recording 3a: ref 13
ref_13 = ref_13[(ref_13['Time (s)'] > 2.9)]
plt.figure(figsize=(12,4))
plt.plot(ref_13['Time (s)'], ref_13['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Reference Recording 3a")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=1.5)
plt.show()
# Rov recording 3a: rov 15
rov_15 = rov_15[(rov_15['Time (s)'] > 1.95)] 
plt.figure(figsize=(12,4))
plt.plot(rov_15['Time (s)'], rov_15['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Roving Recording 3a")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=1)
plt.show()

# Ref recording 4a: ref 16
ref_16 = ref_16[(ref_16['Time (s)'] > 4.1)]
plt.figure(figsize=(12,4))
plt.plot(ref_16['Time (s)'], ref_16['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Reference Recording 4a")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=1.5)
plt.show()
# rov recording 4a: rov 18
rov_18 = rov_18[(rov_18['Time (s)'] > 3.2)]
plt.figure(figsize=(12,4))
plt.plot(rov_18['Time (s)'], rov_18['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Roving Recording 4a")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=3.5)
plt.show()

# Ref recording 5a: ref 19
ref_19 = ref_19[(ref_19['Time (s)'] > 2.4)]
plt.figure(figsize=(12,4))
plt.plot(ref_19['Time (s)'], ref_19['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Reference Recording 5a")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=1.5)
plt.show()
# Rov recording 5a: rov 21
rov_21 = rov_21[(rov_21['Time (s)'] > 1.75)] 
plt.figure(figsize=(12,4))
plt.plot(rov_21['Time (s)'], rov_21['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Roving Recording 5a")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=3.5)
plt.show()

# Recording B's~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Ref recording 2b: ref_10
ref_10 = ref_10[(ref_10['Time (s)'] > 3.3)]
plt.figure(figsize=(12,4))
plt.plot(ref_10['Time (s)'], ref_10['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Reference Recording 2b")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=3.6)
plt.show()
# Rov recording 2b: rov_12
rov_12 = rov_12[(rov_12['Time (s)'] > 1.9)] 
plt.figure(figsize=(12,4))
plt.plot(rov_12['Time (s)'], rov_12['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Roving Recording 2b")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=3.5)
plt.show()

# Ref recording 3b: ref_15
ref_15 = ref_15[(ref_15['Time (s)'] > 2.8)]
plt.figure(figsize=(12,4))
plt.plot(ref_15['Time (s)'], ref_15['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Reference Recording 3b")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=3.6)
plt.show()
# Rov recording 3b: rov_17
rov_17 = rov_17[(rov_17['Time (s)'] > 2)]
plt.figure(figsize=(12,4))
plt.plot(rov_17['Time (s)'], rov_17['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Roving Recording 3b")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=3.5)
plt.show()

# Ref recording 4b: ref_17
ref_17 = ref_17[(ref_17['Time (s)'] > 1.8)]
plt.figure(figsize=(12,4))
plt.plot(ref_17['Time (s)'], ref_17['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Reference Recording 4b")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=3.6)
plt.show()
# Rov recording 4b: rov_19
rov_19 = rov_19[(rov_19['Time (s)'] > 1)]
plt.figure(figsize=(12,4))
plt.plot(rov_19['Time (s)'], rov_19['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Roving Recording 4b")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=1.5)
plt.show()

# Ref recording 5b: ref_20
ref_20 = ref_20[(ref_20['Time (s)'] > 1.5)]
plt.figure(figsize=(12,4))
plt.plot(ref_20['Time (s)'], ref_20['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Reference Recording 5b")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=3.6)
plt.show()
# Rov recording 5b: rov_22
rov_22 = rov_22[(rov_22['Time (s)'] > 0.75)]
plt.figure(figsize=(12,4))
plt.plot(rov_22['Time (s)'], rov_22['Amplitude'])
plt.grid(True, alpha=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Trimmed Roving Recording 5b")
plt.ylim(-0.1, 0.1)
#plt.xlim(right=1.5)
plt.show()

# Add trimmmed measurements to list again, the will replace the previous lists...
run_A_mic_ref = [ref_08, ref_09, ref_13, ref_16, ref_19] # all run A reference measurements
run_A_mic_rov = [rov_10, rov_11, rov_15, rov_18, rov_21] # all run A roving measurements
run_A_all = [ref_08, rov_10, ref_09, rov_11, ref_13, rov_15, ref_16, rov_18, ref_19, rov_21] # ref + rov, ordered by run A number

run_B_mic_ref = [ref_10, ref_15, ref_17, ref_20] # all run B reference measurements
run_B_mic_rov = [rov_12, rov_17, rov_19, rov_22] # all run B roving measurements
run_B_all = [ref_10, rov_12, ref_15, rov_17, ref_17, rov_19, ref_20, rov_22] # ref + rov, ordered by run B number

***
##### Resample Measurements
- Resample vibrometer measurements to match the sample rate of the generated square wave and the exported microphone recordings (44100 Hz)

In [ ]:
# Try signal.resample.poly:
# sample rate of raw vibrometer data: 125000 Hz
# resampling ratio = 1250/441

# Background measurement:
resampled_background = signal.resample_poly(background['Signal1 (m)'], 441, 1250)
# Create correct time array
dt = 1/44100
resampled_time = np.arange(len(resampled_background)) * dt + background['Time1 (s)'].iloc[0]
# Verify:
print(f"Original duration: {background['Time1 (s)'].iloc[-1] - background['Time1 (s)'].iloc[0]:.2f} s")
print(f"Resampled duration: {resampled_time[-1] - resampled_time[0]:.2f} s")
print(f"New sample rate: {1/(resampled_time[1]-resampled_time[0]):.1f} Hz")
print("resampled_time[-1]:", resampled_time[-1])
# Plot:
plt.figure(figsize=(12,4))
plt.plot(resampled_time, resampled_background)
plt.grid(True, alpha=0.4)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (m)")
plt.title("Resampled Background Measurement")
plt.show()

# Initial Measurement (No Glass):
resampled_no_glass = signal.resample_poly(no_glass['Signal2 (m)'], 441, 1250)
# Create correct time array
dt_no_glass = 1/44100
resampled_time_no_glass = np.arange(len(resampled_no_glass)) * dt_no_glass + no_glass['Time2 (s)'].iloc[0]
# Verify:
print(f"Original duration: {no_glass['Time2 (s)'].iloc[-1] - no_glass['Time2 (s)'].iloc[0]:.2f} s")
print(f"Resampled duration: {resampled_time_no_glass[-1] - resampled_time_no_glass[0]:.2f} s")
print(f"New sample rate: {1/(resampled_time_no_glass[1] - resampled_time_no_glass[0]):.1f} Hz")
print("resampled_time[-1]:", resampled_time_no_glass[-1])
# Plot:
plt.figure(figsize=(12,4))
plt.plot(resampled_time_no_glass, resampled_no_glass)
plt.grid(True, alpha=0.4)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (m)")
plt.title("Resampled Square Wave Measurement (No Glass)")
plt.show()

# Run A Measurements:
for i, dframe in enumerate(run_A_measurements):
    time_col_a = f"Time{i+1}a (s)"
    signal_col_a = f"Signal{i+1}a (m)"
    resampled_measurement = signal.resample_poly(dframe[signal_col_a], 441, 1250)
    delta_t = 1/44100
    resampled_time = np.arange(len(resampled_measurement)) * delta_t + dframe[time_col_a].iloc[0]
    # Verify:
    print(f"Original duration: {dframe[time_col_a].iloc[-1] - dframe[time_col_a].iloc[0]:.2f} s")
    print(f"Resampled duration: {resampled_time[-1] - resampled_time[0]:.2f} s")
    print(f"New sample rate: {1/(resampled_time[1]-resampled_time[0]):.1f} Hz")
    print("resampled_time[-1]:", resampled_time[-1])
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(resampled_time, resampled_measurement)
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    plt.title(f"Resampled Measurement {i+1}a")
    plt.show()
    
# Run B Measurements:
for i, dframe in enumerate(run_B_measurements):
    time_col_b = f"Time{i+2}b (s)"
    signal_col_b = f"Signal{i+2}b (m)"
    resampled_measurement = signal.resample_poly(dframe[signal_col_b], 441, 1250)
    delta_t = 1/44100
    resampled_time = np.arange(len(resampled_measurement)) * delta_t + dframe[time_col_b].iloc[0]
    # Verify:
    print(f"Original duration: {dframe[time_col_b].iloc[-1] - dframe[time_col_b].iloc[0]:.2f} s")
    print(f"Resampled duration: {resampled_time[-1] - resampled_time[0]:.2f} s")
    print(f"New sample rate: {1/(resampled_time[1]-resampled_time[0]):.1f} Hz")
    print("resampled_time[-1]:", resampled_time[-1])
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(resampled_time, resampled_measurement)
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    plt.title(f"Resampled Measurement {i+2}b")
    plt.show()

In [ ]:
# Make new dataframes for the resampled measurements:

# Signals:
resampled_background = signal.resample_poly(background['Signal1 (m)'], 441, 1250)
resampled_no_glass = signal.resample_poly(no_glass['Signal2 (m)'], 441, 1250)
# run A resampled:
resampled_1a = signal.resample_poly(run_1a['Signal1a (m)'], 441, 1250)
resampled_2a = signal.resample_poly(run_2a['Signal2a (m)'], 441, 1250)
resampled_3a = signal.resample_poly(run_3a['Signal3a (m)'], 441, 1250)
resampled_4a = signal.resample_poly(run_4a['Signal4a (m)'], 441, 1250)
resampled_5a = signal.resample_poly(run_5a['Signal5a (m)'], 441, 1250)
# run B resampled:
resampled_2b = signal.resample_poly(run_2b['Signal2b (m)'], 441, 1250)
resampled_3b = signal.resample_poly(run_3b['Signal3b (m)'], 441, 1250)
resampled_4b = signal.resample_poly(run_4b['Signal4b (m)'], 441, 1250)
resampled_5b = signal.resample_poly(run_5b['Signal5b (m)'], 441, 1250)

# Times:
dt = 1/44100
resampled_time = np.arange(len(resampled_background)) * dt + background['Time1 (s)'].iloc[0] # for background measurement
resampled_time_no_glass = np.arange(len(resampled_no_glass)) * dt_no_glass + no_glass['Time2 (s)'].iloc[0]
# run A resampled:
resampled_time_1a = np.arange(len(resampled_1a)) * dt + run_1a['Time1a (s)'].iloc[0]
resampled_time_2a = np.arange(len(resampled_2a)) * dt + run_2a['Time2a (s)'].iloc[0]
resampled_time_3a = np.arange(len(resampled_3a)) * dt + run_3a['Time3a (s)'].iloc[0]
resampled_time_4a = np.arange(len(resampled_4a)) * dt + run_4a['Time4a (s)'].iloc[0]
resampled_time_5a = np.arange(len(resampled_5a)) * dt + run_5a['Time5a (s)'].iloc[0]
# run B resampled:
resampled_time_2b = np.arange(len(resampled_2b)) * dt + run_2b['Time2b (s)'].iloc[0]
resampled_time_3b = np.arange(len(resampled_3b)) * dt + run_3b['Time3b (s)'].iloc[0]
resampled_time_4b = np.arange(len(resampled_4b)) * dt + run_4b['Time4b (s)'].iloc[0]
resampled_time_5b = np.arange(len(resampled_5b)) * dt + run_5b['Time5b (s)'].iloc[0]

# Put signals and times into Pandas dataframes:
resampled_background_df = pd.DataFrame({'Resampled Measurement1 (m)': resampled_background, 'Resampled Time1 (s)': resampled_time})
resampled_no_glass_df = pd.DataFrame({'Resampled Measurement2 (m)': resampled_no_glass, 'Resampled Time2 (s)': resampled_time_no_glass})
resampled_1a_df = pd.DataFrame({'Resampled Measurement1a (m)': resampled_1a, 'Resampled Time1a (s)': resampled_time_1a})
resampled_2a_df = pd.DataFrame({'Resampled Measurement2a (m)': resampled_2a, 'Resampled Time2a (s)': resampled_time_2a})
resampled_3a_df = pd.DataFrame({'Resampled Measurement3a (m)': resampled_3a, 'Resampled Time3a (s)': resampled_time_3a})
resampled_4a_df = pd.DataFrame({'Resampled Measurement4a (m)': resampled_4a, 'Resampled Time4a (s)': resampled_time_4a})
resampled_5a_df = pd.DataFrame({'Resampled Measurement5a (m)': resampled_5a, 'Resampled Time5a (s)': resampled_time_5a})
resampled_2b_df = pd.DataFrame({'Resampled Measurement2b (m)': resampled_2b, 'Resampled Time2b (s)': resampled_time_2b})
resampled_3b_df = pd.DataFrame({'Resampled Measurement3b (m)': resampled_3b, 'Resampled Time3b (s)': resampled_time_3b})
resampled_4b_df = pd.DataFrame({'Resampled Measurement4b (m)': resampled_4b, 'Resampled Time4b (s)': resampled_time_4b})
resampled_5b_df = pd.DataFrame({'Resampled Measurement5b (m)': resampled_5b, 'Resampled Time5b (s)': resampled_time_5b})

***
##### Detrend and Shift Measurements to Zero Mean

In [ ]:
resampled_A_measurements = [resampled_1a_df, resampled_2a_df, resampled_3a_df, resampled_4a_df, resampled_5a_df]
resampled_B_measurements = [resampled_2b_df, resampled_3b_df, resampled_4b_df, resampled_5b_df]

# Background:
# Detrend
resampled_background_df['Detrended Measurement1 (m)'] = scipy.signal.detrend(resampled_background_df['Resampled Measurement1 (m)'], type='linear')
# Then shift mean to approx. zero
resampled_background_df['Detrended/Shifted Measurement1 (m)']= resampled_background_df['Detrended Measurement1 (m)'] - np.mean(resampled_background_df['Detrended Measurement1 (m)'])
print(f"Background measurement shifted mean: {np.mean(resampled_background_df['Detrended/Shifted Measurement1 (m)'])}")
# Plot
plt.figure(figsize=(12,4))
plt.plot(resampled_background_df['Resampled Time1 (s)'], resampled_background_df['Detrended/Shifted Measurement1 (m)'])
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (m)")
plt.grid(True,alpha=0.4)
plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
plt.title("Resampled Background Measurement - Detrended and Shifted to Zero Mean")
plt.tight_layout()
plt.show()

# No glass:
# Detrend
resampled_no_glass_df['Detrended Measurement2 (m)'] = scipy.signal.detrend(resampled_no_glass_df['Resampled Measurement2 (m)'], type='linear')
# Then shift meanto approx. zero
resampled_no_glass_df['Detrended/Shifted Measurement2 (m)'] = resampled_no_glass_df['Detrended Measurement2 (m)'] - np.mean(resampled_no_glass_df['Detrended Measurement2 (m)'])
print(f"No glass measurement shifted mean: {np.mean(resampled_no_glass_df['Detrended/Shifted Measurement2 (m)'])}")
# Plot
plt.figure(figsize=(12,4))
plt.plot(resampled_no_glass_df['Resampled Time2 (s)'], resampled_no_glass_df['Detrended/Shifted Measurement2 (m)'])
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (m)")
plt.grid(True,alpha=0.4)
plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
plt.title("Resampled No Glass Measurement - Detrended and Shifted to Zero Mean")
plt.tight_layout()
plt.show()

# run A measurements:
for i, dframe in enumerate(resampled_A_measurements):
    sig_col = f"Resampled Measurement{i+1}a (m)"
    time_col = f"Resampled Time{i+1}a (s)"
    detrended_measurement_col = f"Detrended Measurement{i+1} (m)"
    mean_shifted_col = f"Detrended/Shifted Measurement {i+1} (m)"
   # Detrend
    dframe[detrended_measurement_col] = scipy.signal.detrend(dframe[sig_col], type='linear') 
    # Then Shift Mean to ~Zero (without normalizing)
    dframe[mean_shifted_col] = dframe[detrended_measurement_col] - np.mean(dframe[detrended_measurement_col])
    print(f"M{i+1} Mean:", np.mean(dframe[mean_shifted_col]))
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col], dframe[mean_shifted_col])
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    plt.grid(True,alpha=0.4)
    plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
    plt.title(f"Resampled Measurement {i+1}a - Detrended and Shifted to Zero Mean")
    plt.tight_layout()
    plt.show()

# Run B measurements:
for i, dframe in enumerate(resampled_B_measurements):
    sig_col = f"Resampled Measurement{i+2}b (m)"
    time_col = f"Resampled Time{i+2}b (s)"
    detrended_measurement_col = f"Detrended Measurement{i+2} (m)"
    mean_shifted_col = f"Detrended/Shifted Measurement {i+2} (m)"
   # Detrend
    dframe[detrended_measurement_col] = scipy.signal.detrend(dframe[sig_col], type='linear') 
    # Then Shift Mean to ~Zero (without normalizing)
    dframe[mean_shifted_col] = dframe[detrended_measurement_col] - np.mean(dframe[detrended_measurement_col])
    print(f"M{i+2} Mean:", np.mean(dframe[mean_shifted_col]))
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col], dframe[mean_shifted_col])
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    plt.grid(True, alpha=0.4)
    plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
    plt.title(f"Resampled Measurement {i+2}b - Detrended and Shifted to Zero Mean")
    plt.tight_layout()
    plt.show()

In [ ]:
# microphone recordings
# For run A recordings:
for i, (ref, rov) in enumerate(zip(run_A_mic_ref, run_A_mic_rov)):
    signal_col = "Amplitude"
    time_col= 'Time (s)'
    detrended_col = "Detrended"
    detrended_shifted_col = "Detrended/Shifted"
    # Detrend ref and rov recordings
    ref[detrended_col] = scipy.signal.detrend(ref[signal_col], type='linear') 
    rov[detrended_col] = scipy.signal.detrend(rov[signal_col], type='linear') 
    # Then Shift Mean to ~Zero (without normalizing)
    ref[detrended_shifted_col] = ref[detrended_col] - np.mean(ref[detrended_col])
    print("Ref Mean:", np.mean(ref[detrended_shifted_col]))
    rov[detrended_shifted_col] = rov[detrended_col] - np.mean(rov[detrended_col])
    print("Rov Mean:", np.mean(rov[detrended_shifted_col]))
    
    # Plot reference:
    plt.figure(figsize=(12,4))
    plt.plot(ref[time_col], ref[detrended_shifted_col]) # detrended and shifted to ~zero mean
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.ylim(-0.1, 0.1)
    plt.title(f"Reference Recording {i+1}a - Detrended and Shifted to Zero Mean")
    plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
    plt.show()
    
    # Plot roving:
    plt.figure(figsize=(12,4))
    plt.plot(rov[time_col], rov[detrended_shifted_col]) # detrended and shifted to ~zero mean
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.ylim(-0.1, 0.1)
    plt.title(f"Roving Recording {i+1}a - Detrended and Shifted to Zero Mean")
    plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
    plt.show()

# For run B recordings:
for i, (ref, rov) in enumerate(zip(run_B_mic_ref, run_B_mic_rov)):
    signal_col = 'Amplitude'
    time_col = 'Time (s)'
    detrended_col = "Detrended"
    detrended_shifted_col = "Detrended/Shifted"
    # Detrend ref and rov recordings
    ref[detrended_col] = scipy.signal.detrend(ref[signal_col], type='linear') 
    rov[detrended_col] = scipy.signal.detrend(rov[signal_col], type='linear') 
    # Then Shift Mean to ~Zero (without normalizing)
    ref[detrended_shifted_col] = ref[detrended_col] - np.mean(ref[detrended_col])
    print("Ref Mean:", np.mean(ref[detrended_shifted_col]))
    rov[detrended_shifted_col] = rov[detrended_col] - np.mean(rov[detrended_col])
    print("Rov Mean:", np.mean(rov[detrended_shifted_col]))

    # Plot reference:
    plt.figure(figsize=(12,4))
    plt.plot(ref[time_col], ref[signal_col])
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.ylim(-0.1, 0.1)
    plt.title(f"Reference Recording {i+2}b - Detrended and Shifted to Zero Mean")
    plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
    plt.show()

    # Plot roving:
    plt.figure(figsize=(12,4))
    plt.plot(rov[time_col], rov[signal_col])
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.ylim(-0.1, 0.1)
    plt.title(f"Roving Recording {i+2}b - Detrended and Shifted to Zero Mean")
    plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
    plt.show()

***
##### Taper Measurements
- A Hann window is used for tapering
- The taper is set to start at 10% from the data edges, but this can be adjusted with the taper_fraction parameter

In [ ]:
def apply_hann_window(data, column, taper_fraction = 0.1):
    """Taper edges of time series with Hann window
    taper_fraction = 0.1 will start taper at 10% from edges"""
    n= len(data)
    taper_length = int(n * taper_fraction)
    
    # Create a copy of the data
    windowed_data = data[column].copy()
    windowed_data = windowed_data.astype(float)
    
    # Apply Hann window to the left edge
    left_window = np.hanning(2 * taper_length)[:taper_length]
    windowed_data.iloc[:taper_length] *= left_window
    
    # Apply Hann window to the right edge
    right_window = np.hanning(2 * taper_length)[taper_length:]
    windowed_data.iloc[-taper_length:] *= right_window
    
    return windowed_data
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# VIBROMETER RECORDINGS:
print("Vibrometer Measurements:")
for i, dframe in enumerate(resampled_A_measurements):
    time_col = f"Resampled Time{i+1}a (s)"
    mean_shifted_col = f"Detrended/Shifted Measurement {i+1} (m)"
    tapered_result_a = apply_hann_window(dframe, mean_shifted_col, taper_fraction=0.1)
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col], dframe[mean_shifted_col], label='full')
    plt.plot(dframe[time_col], tapered_result_a, label='tapered')
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    plt.grid(True, alpha=0.4)
    plt.legend(loc='upper left')
    plt.title(f"Resampled Measurement {i+1}a - Full & Tapered")
    plt.tight_layout()
    plt.show()

for i, dframe in enumerate(resampled_B_measurements):
    time_col = f"Resampled Time{i+2}b (s)"
    mean_shifted_col = f"Detrended/Shifted Measurement {i+2} (m)"
    tapered_result_b = apply_hann_window(dframe, mean_shifted_col, taper_fraction=0.1)
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col], dframe[mean_shifted_col], label='full')
    plt.plot(dframe[time_col], tapered_result_b, label='tapered')
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    plt.grid(True, alpha=0.4)
    plt.legend(loc='upper left')
    plt.title(f"Resampled Measurement {i+2}b - Full & Tapered")
    plt.tight_layout()
    plt.show()
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# MICROPHONE RECORDINGS:
print("Microphone Recordings:")
for i, (ref, rov) in enumerate(zip(run_A_mic_ref, run_A_mic_rov)):
    time_col= 'Time (s)'
    detrended_shifted_col = "Detrended/Shifted"
    tapered_ref = apply_hann_window(ref, detrended_shifted_col, taper_fraction=0.1)
    tapered_rov = apply_hann_window(rov, detrended_shifted_col, taper_fraction=0.1)
    
    # Plot reference:
    plt.figure(figsize=(12,4))
    plt.plot(ref[time_col], ref[detrended_shifted_col], label='full') # detrended and shifted to ~zero mean
    plt.plot(ref[time_col], tapered_ref, label='tapered') # tapered version
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.legend(loc='upper left')
    plt.title(f"Reference Recording {i+1}a - Full & Tapered")
    plt.show()
    
    # Plot roving:
    plt.figure(figsize=(12,4))
    plt.plot(rov[time_col], rov[detrended_shifted_col], label='full') # detrended and shifted to ~zero mean
    plt.plot(rov[time_col], tapered_rov, label='tapered')
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.legend(loc='upper left')
    plt.title(f"Roving Recording {i+1}a - Full & Tapered")
    plt.show()

for i, (ref, rov) in enumerate(zip(run_B_mic_ref, run_B_mic_rov)):
    time_col= 'Time (s)'
    detrended_shifted_col = "Detrended/Shifted"
    tapered_ref = apply_hann_window(ref, detrended_shifted_col, taper_fraction=0.1)
    tapered_rov = apply_hann_window(rov, detrended_shifted_col, taper_fraction=0.1)
    
    # Plot reference:
    plt.figure(figsize=(12,4))
    plt.plot(ref[time_col], ref[detrended_shifted_col], label='full') # detrended and shifted to ~zero mean
    plt.plot(ref[time_col], tapered_ref, label='tapered') # tapered version
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.legend(loc='upper left')
    plt.title(f"Reference Recording {i+2}b - Full & Tapered")
    plt.show()
    
    # Plot roving:
    plt.figure(figsize=(12,4))
    plt.plot(rov[time_col], rov[detrended_shifted_col], label='full') # detrended and shifted to ~zero mean
    plt.plot(rov[time_col], tapered_rov, label='tapered')
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.legend(loc='upper left')
    plt.title(f"Roving Recording {i+2}b - Full & Tapered")
    plt.show()

***
##### Averaged FFT:
- Make sure 'segment_size' is not larger than the length of the data

In [ ]:
def plot_averaged_fft(data, column, sr, segment_size, title=''):
    """Plot FFT with frequency bin averaging 
    data: pandas DataFrame
    column: relevant data column from data, string
    sr: Sampling rate in Hz
    segment_size:segment size for averaging, int
    title: plot title, str"""
    # Goal: decrease frequency resolution of the FFT
    # Signal parameters
    T = 1 / sr  # Sampling period
    # signal x:
    x = data[column].values
    N = len(x)
    
    # Parameters for averaging
    overlap = 0.5   # 50% overlap
    step_size = int(segment_size * (1 - overlap))
    
    # Calculate number of segments
    num_segments = (N - segment_size) // step_size + 1
    
    # Initialize array to store summed FFT results
    avg_fft_magnitude = np.zeros(segment_size // 2 + 1) # only need one side for real input
    
    for i in range(num_segments):
        start = i * step_size
        end = start + segment_size
        segment = x[start:end]
        
        # apply a Hanning window to the segment to reduce spectral leakage
        window = np.hanning(segment_size)
        windowed_segment = segment * window
        
        # Compute FFT of the segment
        segment_fft = scipy.fft.rfft(windowed_segment, segment_size)
        
        # Get the magnitude of the positive frequency side
        #segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1]
        segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1] / np.sum(window) # window sum is for amplitude correction
        
        # Accumulate the magnitudes
        #avg_fft_magnitude += segment_magnitude 
        avg_fft_magnitude += segment_magnitude ** 2 #accumulate power
    
    # Calculate the mean magnitude
    #avg_fft_magnitude /= num_segments
    avg_fft_magnitude = np.sqrt(avg_fft_magnitude / num_segments) # don't take square root if want power spectral density instead
    
    # Get the frequency bins
    freqs = scipy.fft.rfftfreq(segment_size, T)[:segment_size // 2 + 1]
    
    # Plot the averaged frequency spectrum
    plt.figure(figsize=(12, 6))
    #plt.plot(freqs, avg_fft_magnitude)
    plt.loglog(freqs, avg_fft_magnitude)
    plt.xlabel('Frequency (Hz)', fontsize=12)
    plt.ylabel('Averaged FFT Magnitude', fontsize=12)
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    plt.show()

    return freqs, avg_fft_magnitude
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# VIBROMETER RECORDINGS:
print("Vibrometer Measurements:")
for i, dframe in enumerate(resampled_A_measurements):
    time_col = f"Resampled Time{i+1}a (s)"
    mean_shifted_col = f"Detrended/Shifted Measurement {i+1} (m)"
    titles = f"Averaged FFT {i+1}a"
    freqs, avg_mags = plot_averaged_fft(dframe, mean_shifted_col, 44100, segment_size=4096, title=titles)

for i, dframe in enumerate(resampled_B_measurements):
    time_col = f"Resampled Time{i+2}b (s)"
    mean_shifted_col = f"Detrended/Shifted Measurement {i+2} (m)"
    titles = f"Averaged FFT {i+2}b"
    freqs, avg_mags = plot_averaged_fft(dframe, mean_shifted_col, 44100, segment_size=4096, title=titles)

# MICROPHONE RECORDINGS:~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Microphone Recordings:")
for i, (ref, rov) in enumerate(zip(run_A_mic_ref, run_A_mic_rov)):
    time_col= 'Time (s)'
    detrended_shifted_col = "Detrended/Shifted"
    ref_titles = f"Averaged FFT - Ref {i+1}a"
    rov_titles = f"Averaged FFT - Rov {i+1}a"
    freqs_ref, avg_mags_ref = plot_averaged_fft(ref, detrended_shifted_col, 44100, segment_size=4096, title=ref_titles)
    freqs_rov, avg_mags_rov = plot_averaged_fft(rov, detrended_shifted_col, 44100, segment_size=4096, title=rov_titles)

for i, (ref, rov) in enumerate(zip(run_B_mic_ref, run_B_mic_rov)):
    time_col= 'Time (s)'
    detrended_shifted_col = "Detrended/Shifted"
    ref_titles = f"Averaged FFT - Ref {i+2}b"
    rov_titles = f"Averaged FFT - Rov {i+2}b"
    freqs_ref, avg_mags_ref = plot_averaged_fft(ref, detrended_shifted_col, 44100, segment_size=4096, title=ref_titles)
    freqs_rov, avg_mags_rov = plot_averaged_fft(rov, detrended_shifted_col, 44100, segment_size=4096, title=rov_titles)

##### Power Spectral Density With Welch's Method:

In [ ]:
def welch_psd(data, column, sr, title, segment_size, window='hann'):
    """data: pandas DataFrame
    column: relevant data column from data, str
    sr: Sampling rate in Hz
    title: title for PSD plot, str
    segment_size: length of each segment
    window: desired window to use"""
    
    overlap = segment_size//2 # number of points to overlap between segments, here is 50% overlap
    freqs, psd = welch(data[column].values, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # Plot:
    plt.figure(figsize=(12,6))
    #plt.plot(freqs,psd)
    plt.loglog(freqs, psd)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD")
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    plt.show()

    return freqs, psd
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# VIBROMETER RECORDINGS:
print("Vibrometer Measurements:")
for i, dframe in enumerate(resampled_A_measurements):
    time_col = f"Resampled Time{i+1}a (s)"
    mean_shifted_col = f"Detrended/Shifted Measurement {i+1} (m)"
    titles = f"Welch PSD - Measurement {i+1}a"
    freqs, psd = welch_psd(dframe, mean_shifted_col, 44100, title=titles, segment_size=2096, window='hann')
    
for i, dframe in enumerate(resampled_B_measurements):
    time_col = f"Resampled Time{i+2}b (s)"
    mean_shifted_col = f"Detrended/Shifted Measurement {i+2} (m)"
    titles = f"Welch PSD - Measurement {i+2}b"
    freqs, psd = welch_psd(dframe, mean_shifted_col, 44100, title=titles, segment_size=2096, window='hann')

# MICROPHONE RECORDINGS:~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Microphone Recordings:")
for i, (ref, rov) in enumerate(zip(run_A_mic_ref, run_A_mic_rov)):
    time_col= 'Time (s)'
    detrended_shifted_col = "Detrended/Shifted"
    ref_titles = f"Welch PSD - Ref {i+1}a"
    rov_titles = f"Welch PSD - Rov {i+1}a"
    freqs_ref, psd_ref = welch_psd(ref, detrended_shifted_col, 44100, segment_size=4096, title=ref_titles)
    freqs_rov, psd_rov = welch_psd(rov, detrended_shifted_col, 44100, segment_size=4096, title=rov_titles)
    
for i, (ref, rov) in enumerate(zip(run_B_mic_ref, run_B_mic_rov)):
    time_col= 'Time (s)'
    detrended_shifted_col = "Detrended/Shifted"
    ref_titles = f"Welch PSD - Ref {i+2}b"
    rov_titles = f"Welch PSD - Rov {i+2}b"
    freqs_ref, psd_ref = welch_psd(ref, detrended_shifted_col, 44100, segment_size=4096, title=ref_titles)
    freqs_rov, psd_rov = welch_psd(rov, detrended_shifted_col, 44100, segment_size=4096, title=rov_titles)

#### Estimating Transfer Function:
| Transfer Function | Run | Input | Output | Physical meaning |
|---|---|---|---|---|
| H_glass | A | rov mic (vibrometer side) | vibrometer measurement | Acoustic-to-mechanical coupling of glass |
| H_total | A | ref mic (at speaker face) | vibrometer measurement | Full system response |
| H_propagation | B | ref mic (at speaker face) | rov mic (at glass, speaker side) | Propagation through air gap |

Coherence > 0.9 across the frequency band of interest indicates a reliable estimate.

In [ ]:
def welch_transfer_function(input_data, input_column, output_data, output_column, sr, title, segment_size, lowcut, highcut, window='hann'):
    """Estimate transfer function using Welch's method.
    input_data/output_data: pandas DataFrames
    input_column/output_column: relevant column names, str
    sr: sampling rate in Hz
    title: title for plots, str
    segment_size: length of each segment (nperseg), controls frequency resolution
    lowcut: lowest frequency for bandpass filter, int
    highcut: highest frequency for bandpass filter, int
    window: window function to use
    
    Returns: freqs, H, magnitudes, phases_deg, coherence"""
    overlap = segment_size // 2
    x = input_data[input_column].values
    y = output_data[output_column].values

    # New change: Apply bandpass filter~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    def bandpass(signal, low, high, fs, order=4):
        sos = butter(order, [low, high], btype='band', fs=fs, output='sos')
        return sosfilt(sos, signal)

    x = bandpass(x, lowcut, highcut, fs=sr)
    y = bandpass(y, lowcut, highcut, fs=sr)

    # power spectral density of input~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    freqs, Pxx = welch(x, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # Cross-power spectral density (output vs input)
    #freqs, Pyx = csd(y, x, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')
    freqs, Pyx = csd(x, y, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # H1 estimator: H(f) = Pyx/Pxx
    H = Pyx/Pxx
    magnitudes = np.abs(H)
    phases_deg = np.degrees(np.angle(H))

    # Coherence: quality metric, close to 1.0 indicates good estimate
    freqs_coh, Cxy = coherence(x, y, fs=sr, nperseg=segment_size, noverlap=overlap, window=window)

    ###~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Apply mask to BOTH coherence arrays together
    coh_mask = (freqs_coh >= lowcut) & (freqs_coh <= highcut) # should be same values used with bandpass function
    freqs_coh = freqs_coh[coh_mask]
    Cxy = Cxy[coh_mask]
    
    # And for the main spectrum:
    band_mask = (freqs >= lowcut) & (freqs <= highcut) # should be same values used with bandpass function
    freqs = freqs[band_mask]
    magnitudes = magnitudes[band_mask]
    phases_deg = phases_deg[band_mask]
    H = H[band_mask]

    # Plots
    fig, axes = plt.subplots(3, 1, figsize=(12, 12))

    axes[0].loglog(freqs, magnitudes) 
    axes[0].set_xlabel("Frequency (Hz)")
    axes[0].set_ylabel("Magnitude |H(f)|")
    axes[0].set_title(f"{title} - Transfer Function Magnitude")
    axes[0].grid(True, which='both', alpha=0.5)

    #axes[1].semilogx(freqs, phases_deg)
    axes[1].plot(freqs, phases_deg)
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("Phase (degrees)")
    axes[1].set_title(f"{title} - Transfer Function Phase")
    axes[1].grid(True, which='both', alpha=0.5)
    
    # Add plot for coherence
    axes[2].semilogx(freqs_coh, Cxy)
    axes[2].set_xlabel("Frequency (Hz)")
    axes[2].set_ylabel("Coherence")
    axes[2].set_title(f"{title} - Coherence")
    axes[2].set_ylim(0, 1.1)
    axes[2].axhline(0.9, color='r', linestyle='--', alpha=0.5, label='0.9 threshold')
    axes[2].legend()
    axes[2].grid(True, which='both', alpha=0.5)

    plt.tight_layout()
    plt.show()

    return freqs, H, magnitudes, phases_deg, Cxy

In [ ]:
welch_transfer_function(no_glass, 'Signal2 (m)', run_1a, 'Signal1a (m)', 125000, 'H_total for Position 1', 125000//4, 80, 520)
# trying with just vibrometer measurements, no microphone